# Tech Challenge - Predicao de Cancer de Mama (Wisconsin Diagnostic)

## Objetivo

Desenvolver um modelo de **Classificacao Binaria** para apoiar o diagnostico de **cancer de mama**, com base em caracteristicas de nucleos celulares extraidas de imagens de PAAF (Puncao Aspirativa por Agulha Fina).

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data) / UCI ML Repository

**Variavel alvo:** `diagnosis` (0 = Benigno, 1 = Maligno)

> Ferramenta de apoio a decisao. Nao substitui avaliacao medica, mamografia nem biopsia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
    f1_score,
)

%matplotlib inline

RANDOM_STATE = 42
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

## 1. Carregamento dos dados

In [ ]:
df = pd.read_csv('breast_cancer_wisconsin.csv')
df['diagnosis'] = df['diagnosis'].map({'B': 0, 'M': 1})

print(df.shape)
print(df.info())
df.head()

## 2. Analise Exploratoria (EDA)

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
TARGET = 'diagnosis'
target_counts = df[TARGET].value_counts().sort_index()
print(target_counts.rename({0: 'Benigno', 1: 'Maligno'}))
print(f'Taxa maligna: {df[TARGET].mean():.1%}')

sns.displot(data=df[TARGET])
plt.title('Distribuicao da variavel alvo (diagnosis)')
plt.show()

In [ ]:
df.select_dtypes(include='number').describe().T

In [ ]:
df.select_dtypes(include='number').head()

In [ ]:
# Histograma de todas as variaveis numericas
numeric_cols = [c for c in df.columns if c not in ['id']]
ncols = 4
nrows = int(np.ceil(len(numeric_cols) / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * 3))
axes = np.array(axes).flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(col, fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Distribuicao de todas as variaveis numericas', y=1.01)
plt.tight_layout()
plt.show()

### Graficos de dispersao (pairplot)

Subconjunto das 10 medidas `mean_*` para legibilidade visual.

In [ ]:
mean_cols = [c for c in df.columns if c.startswith('mean ')]
df_pair = df[mean_cols + [TARGET]]

sns.pairplot(
    df_pair,
    hue=TARGET,
    corner=True,
    diag_kind='hist',
    plot_kws={'alpha': 0.4, 's': 15},
    height=1.4,
)
plt.show()

In [ ]:
correlacao = df.select_dtypes(include='number').drop(columns=['id']).corr()
plt.figure(figsize=(14, 12))
sns.heatmap(correlacao, cmap='coolwarm', center=0)
plt.title('Matriz de correlacao')
plt.show()

## 3. Pre-processamento e definicao das features

In [ ]:
FEATURE_COLS = [col for col in df.columns if col not in ['id', TARGET]]

x = df[FEATURE_COLS]
y = df[TARGET].astype(int)

print(f'Total de features: {len(FEATURE_COLS)}')
print('Distribuicao do alvo:')
print(y.value_counts().rename({0: 'Benigno', 1: 'Maligno'}))

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), FEATURE_COLS),
    ]
)

## 4. Separacao treino / teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)

## 5. Treinar Modelo - Regressao Logistica

In [ ]:
model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])

model_lr.fit(X_train, y_train)
predictions_lr = model_lr.predict(X_test)

## 6. Avaliacao de um modelo de classificacao

In [ ]:
resultado = pd.DataFrame(
    zip(predictions_lr, y_test),
    columns=['prediction_lr', 'target']
)
print(resultado.shape)
resultado.head(20)

In [ ]:
resultado.loc[resultado.prediction_lr != resultado.target].shape

## Matriz de confusao

- **VP:** Maligno previsto corretamente
- **VN:** Benigno previsto corretamente
- **FP:** Falso alarme (benigno classificado como maligno)
- **FN:** Falso negativo (maligno classificado como benigno)

In [ ]:
confusion_matrix_lr = confusion_matrix(y_test, predictions_lr)
print(confusion_matrix_lr)

cm_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_lr, display_labels=[0, 1])
cm_display.plot()
plt.title('Matriz de Confusao - Regressao Logistica')
plt.show()

In [ ]:
print(classification_report(y_test, predictions_lr, target_names=['Benigno', 'Maligno']))

## 7. Comparacao de algoritmos

In [ ]:
classifiers = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
    ('Decision Tree', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
    ('Random Forest', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE)),
    ('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=7, weights='distance')),
]

resultados = []

for model_name, classifier in classifiers:
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    resultados.append({
        'Modelo': model_name,
        'Accuracy': acc,
        'Recall': rec,
        'F1-Score': f1,
        'Pipeline': pipeline,
    })

    print(f'{model_name}: Accuracy={acc:.2f} | Recall={rec:.2f} | F1={f1:.2f}')

In [ ]:
resultado_df = pd.DataFrame(resultados).drop(columns='Pipeline')
resultado_df = resultado_df.sort_values(by='F1-Score', ascending=False)
resultado_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=resultado_df, x='F1-Score', y='Modelo')
plt.title('Comparacao de modelos (F1-Score)')
plt.tight_layout()
plt.show()

## 8. Salvamento da pipeline para API REST

Melhor modelo pelo F1-Score salvo com `joblib` + metadados.

In [ ]:
melhor = max(resultados, key=lambda r: r['F1-Score'])
best_pipeline = melhor['Pipeline']
best_name = melhor['Modelo']

print(f'Melhor modelo: {best_name}')
best_pipeline.fit(X_train, y_train)

MODEL_PATH = MODELS_DIR / 'pipeline_breast_cancer.pkl'
METADATA_PATH = MODELS_DIR / 'pipeline_metadata.pkl'

joblib.dump(best_pipeline, MODEL_PATH)
joblib.dump({
    'model_name': best_name,
    'feature_columns': FEATURE_COLS,
    'target_column': TARGET,
    'target_labels': {0: 'Benigno', 1: 'Maligno'},
    'dataset': 'breast_cancer_wisconsin.csv',
}, METADATA_PATH)

print(f'Pipeline salva em: {MODEL_PATH.resolve()}')
print(f'Metadados salvos em: {METADATA_PATH.resolve()}')

## 9. Teste de carga da pipeline (simulando API REST)

In [ ]:
pipeline_carregada = joblib.load(MODEL_PATH)
metadata = joblib.load(METADATA_PATH)

amostra = X_test.head(3)
predicoes = pipeline_carregada.predict(amostra)
probabilidades = pipeline_carregada.predict_proba(amostra)

resposta_api = pd.DataFrame({
    'predicao': predicoes,
    'probabilidade_maligno': probabilidades[:, 1],
    'label': [metadata['target_labels'][p] for p in predicoes],
})

print('Modelo:', metadata['model_name'])
print('Colunas esperadas:', len(metadata['feature_columns']), 'features')
resposta_api

# 10. Relatorio Tecnico

Ver arquivo `RELATORIO_TECNICO.md`.